# تحليل مؤشرات الطلب على الطعام

يقرأ هذا الدفتر ملف `data/processed/food_demand_train_merged.csv` للقراءة فقط. يتحقق من الأعمدة والمفقودات والتكرارات ومفاتيح المشاهدات، ثم يحسب الطلب الأسبوعي وتركيز الطلب حسب الفئة والوجبة وأداء أنواع المراكز والإيراد والخصم والترويج والتذبذب.

لا تُحذف القيم الشاذة ولا تُملأ الأسابيع غير المسجلة. التذبذب لكل وجبة محسوب من إجمالي الطلب المرصود للوجبة في كل أسبوع عبر المراكز. ونسبة الترويج الرئيسية هي نسبة الصفوف التي ظهر فيها البريد الترويجي أو الصفحة الرئيسية؛ وتُعرض النسبتان منفصلتين أيضًا. النتائج وصفية ولا تثبت السببية.

وعد فؤاد: تحميل البيانات المدمجة، والتحقق من الأعمدة الأساسية ونطاق الأسابيع قبل بدء التحليل.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_RELATIVE_PATH = Path('data') / 'processed' / 'food_demand_train_merged.csv'
candidate_paths = [Path.cwd() / DATA_RELATIVE_PATH, Path.cwd().parent / DATA_RELATIVE_PATH]
DATA_PATH = next((path for path in candidate_paths if path.is_file()), None)
if DATA_PATH is None:
    searched = '\n'.join(str(path) for path in candidate_paths)
    raise FileNotFoundError(f'لم يُعثر على ملف البيانات. المسارات المفحوصة:\n{searched}')

df = pd.read_csv(DATA_PATH)
required_columns = {
    'week', 'center_id', 'meal_id', 'num_orders', 'checkout_price',
    'base_price', 'emailer_for_promotion', 'homepage_featured',
    'center_type', 'category',
}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f'أعمدة مطلوبة غير موجودة: {missing_columns}')

print(f'ملف البيانات: {DATA_PATH.resolve()}')
print(f'عدد الصفوف: {len(df):,}؛ عدد الأعمدة: {len(df.columns)}')

ملف البيانات: C:\Users\hp\Desktop\food-demand-analysis\data\processed\food_demand_train_merged.csv
عدد الصفوف: 456,548؛ عدد الأعمدة: 15


وعد فؤاد: فحص القيم المفقودة والتكرارات ومفاتيح الملاحظات لتحديد مشكلات جودة البيانات.


In [2]:
missing_by_column = df.isna().sum()
exact_duplicate_rows = int(df.duplicated().sum())
observation_key = ['week', 'center_id', 'meal_id']
duplicate_observation_keys = int(df.duplicated(observation_key).sum())

for column in ['week', 'num_orders', 'checkout_price', 'base_price']:
    df[column] = pd.to_numeric(df[column], errors='raise')

week_min = int(df['week'].min())
week_max = int(df['week'].max())
expected_weeks = set(range(week_min, week_max + 1))
observed_weeks = set(df['week'].astype(int).unique())
missing_global_weeks = sorted(expected_weeks - observed_weeks)
num_orders_sum = int(df['num_orders'].sum())

quality_summary = pd.Series({
    'الصفوف': len(df),
    'مجموع num_orders': num_orders_sum,
    'الخلايا المفقودة': int(missing_by_column.sum()),
    'الصفوف المكررة بالكامل': exact_duplicate_rows,
    'تكرار مفتاح الأسبوع-المركز-الوجبة': duplicate_observation_keys,
    'الأسابيع المفقودة على مستوى الملف': len(missing_global_weeks),
})
display(quality_summary.to_frame('القيمة'))
if missing_by_column.any():
    display(missing_by_column[missing_by_column > 0].to_frame('الخلايا المفقودة'))
if missing_global_weeks:
    print('الأسابيع غير الموجودة:', missing_global_weeks)
if exact_duplicate_rows or duplicate_observation_keys:
    raise ValueError('توجد تكرارات في المدخل؛ أوقف التحليل ولا تحذفها تلقائيًا.')
if not np.isfinite(df['num_orders']).all():
    raise ValueError('تحتوي num_orders على قيم غير عددية منتهية.')
if (df['num_orders'] < 0).any():
    raise ValueError('تحتوي num_orders على قيم سالبة؛ يلزم فحصها قبل اعتماد المؤشرات.')

,القيمة
الصفوف,456548
مجموع num_orders,119557485
الخلايا المفقودة,0
الصفوف المكررة بالكامل,0
تكرار مفتاح الأسبوع-المركز-الوجبة,0
الأسابيع المفقودة على مستوى الملف,0


وعد فؤاد: حساب إجمالي الطلب لكل أسبوع ومتوسطه وإبراز اتجاهه عبر الأسابيع.


In [3]:
weekly_orders = df.groupby('week', sort=True)['num_orders'].sum().rename('إجمالي الطلبات')
weekly_mean = float(weekly_orders.mean())
first_week = int(weekly_orders.index.min())
last_week = int(weekly_orders.index.max())
first_week_orders = int(weekly_orders.loc[first_week])
last_week_orders = int(weekly_orders.loc[last_week])
endpoint_change_pct = (
    (last_week_orders - first_week_orders) / first_week_orders * 100
    if first_week_orders else np.nan
)

weekly_summary = pd.Series({
    'عدد الأسابيع المرصودة': len(weekly_orders),
    'أول أسبوع وإجمالي طلباته': f'{first_week}: {first_week_orders:,}',
    'آخر أسبوع وإجمالي طلباته': f'{last_week}: {last_week_orders:,}',
    'متوسط الطلب الأسبوعي': weekly_mean,
    'أقل إجمالي أسبوعي': int(weekly_orders.min()),
    'أعلى إجمالي أسبوعي': int(weekly_orders.max()),
    'التغير بين أول وآخر أسبوع (%)': endpoint_change_pct,
})
display(weekly_summary.to_frame('القيمة'))
display(weekly_orders.to_frame())
assert int(weekly_orders.sum()) == num_orders_sum, 'مجموع الطلب الأسبوعي لا يطابق مجموع num_orders.'

,القيمة
عدد الأسابيع المرصودة,145
أول أسبوع وإجمالي طلباته,"1: 792,261"
آخر أسبوع وإجمالي طلباته,"145: 671,926"
متوسط الطلب الأسبوعي,824534.37931
أقل إجمالي أسبوعي,380065
أعلى إجمالي أسبوعي,1303457
التغير بين أول وآخر أسبوع (%),-15.188808


,إجمالي الطلبات
week,
1,792261
2,787084
3,695262
4,743529
5,1198675
...,...
141,776240
142,739661
143,759926


وعد فؤاد: إعداد جداول مرتبة لمساهمة الفئات والوجبات في إجمالي الطلب ونسبة التراكم.


In [4]:
def ranked_share_table(frame, group_column):
    result = (
        frame.groupby(group_column, dropna=False)['num_orders']
        .sum()
        .sort_values(ascending=False)
        .rename('إجمالي الطلبات')
        .to_frame()
    )
    result['حصة الطلب (%)'] = result['إجمالي الطلبات'] / num_orders_sum * 100
    result['الحصة التراكمية (%)'] = result['حصة الطلب (%)'].cumsum()
    count_to_80 = int((result['الحصة التراكمية (%)'] < 80).sum()) + 1
    return result, count_to_80

category_share, categories_to_80 = ranked_share_table(df, 'category')
meal_share, meals_to_80 = ranked_share_table(df, 'meal_id')

print(f'الفئات اللازمة لبلوغ 80% من الطلب: {categories_to_80} من {len(category_share)}')
display(category_share.iloc[:categories_to_80])
print(f'الوجبات اللازمة لبلوغ 80% من الطلب: {meals_to_80} من {len(meal_share)}')
display(meal_share.iloc[:meals_to_80])
assert np.isclose(category_share['حصة الطلب (%)'].sum(), 100.0)
assert np.isclose(meal_share['حصة الطلب (%)'].sum(), 100.0)

الفئات اللازمة لبلوغ 80% من الطلب: 5 من 14


,إجمالي الطلبات,حصة الطلب (%),الحصة التراكمية (%)
category,,,
Beverages,40480525,33.858629,33.858629
Rice Bowl,20874063,17.459436,51.318065
Sandwich,17636782,14.751717,66.069782
Salad,10944336,9.154037,75.223819
Pizza,7383720,6.175874,81.399693


الوجبات اللازمة لبلوغ 80% من الطلب: 22 من 51


,إجمالي الطلبات,حصة الطلب (%),الحصة التراكمية (%)
meal_id,,,
2290,9770303,8.172055,8.172055
1885,8346246,6.980948,15.153003
1754,6818059,5.702745,20.855748
1993,6656856,5.567912,26.423661
1109,6367773,5.326118,31.749779
2707,5932792,4.962292,36.712071
1971,5728620,4.791519,41.503591
2826,5090103,4.257452,45.761043
1727,4735987,3.961263,49.722306


وعد فؤاد: تلخيص الطلب حسب مركز التجهيز ونوعه لمقارنة أحجام الطلب بين المراكز.


In [5]:
center_totals = (
    df.groupby(['center_id', 'center_type'], dropna=False)['num_orders']
    .sum()
    .rename('إجمالي الطلبات المرصودة')
    .reset_index()
)
center_weekly = (
    df.groupby(['center_id', 'center_type', 'week'], dropna=False)['num_orders']
    .sum()
    .groupby(['center_id', 'center_type'], dropna=False)
    .mean()
    .rename('متوسط الطلب الأسبوعي المرصود لكل مركز')
    .reset_index()
)
center_type_summary = (
    center_totals.groupby('center_type', dropna=False)
    .agg(عدد_المراكز=('center_id', 'nunique'), إجمالي_الطلبات=('إجمالي الطلبات المرصودة', 'sum'), متوسط_إجمالي_المركز=('إجمالي الطلبات المرصودة', 'mean'))
    .join(center_weekly.groupby('center_type', dropna=False)['متوسط الطلب الأسبوعي المرصود لكل مركز'].mean())
    .sort_values('متوسط الطلب الأسبوعي المرصود لكل مركز', ascending=False)
)
display(center_type_summary)
assert int(center_totals['إجمالي الطلبات المرصودة'].sum()) == num_orders_sum

,عدد_المراكز,إجمالي_الطلبات,متوسط_إجمالي_المركز,متوسط الطلب الأسبوعي المرصود لكل مركز
center_type,,,,
TYPE_B,15,29996073,1.999738e+06,13982.632701
TYPE_A,43,68978517,1.604152e+06,11063.114194
TYPE_C,19,20582895,1.083310e+06,7471.105263


وعد فؤاد: حساب مؤشرات الطلب التجارية، ومنها الإيرادات والخصوم ونسب الترويج.


In [ ]:
revenue = float((df['checkout_price'] * df['num_orders']).sum())
valid_base_price = df['base_price'].notna() & (df['base_price'] != 0)
discount_rate = (df.loc[valid_base_price, 'base_price'] - df.loc[valid_base_price, 'checkout_price']) / df.loc[valid_base_price, 'base_price']
mean_discount_pct = float(discount_rate.mean() * 100)
emailer_rate_pct = float(df['emailer_for_promotion'].eq(1).mean() * 100)
homepage_rate_pct = float(df['homepage_featured'].eq(1).mean() * 100)
any_promotion_rate_pct = float((df['emailer_for_promotion'].eq(1) | df['homepage_featured'].eq(1)).mean() * 100)

commercial_kpis = pd.Series({
    'إجمالي الطلبات (وحدة طلب)': num_orders_sum,
    'الإيرادات المرصودة (checkout_price × num_orders)': revenue,
    'متوسط الخصم (%) على الصفوف ذات base_price غير الصفري': mean_discount_pct,
    'نسبة صفوف البريد الترويجي (%)': emailer_rate_pct,
    'نسبة صفوف الظهور في الصفحة الرئيسية (%)': homepage_rate_pct,
    'نسبة الصفوف التي بها أي من الترويجين (%)': any_promotion_rate_pct,
    'عدد الصفوف المستخدمة في متوسط الخصم': int(valid_base_price.sum()),
    'صفوف base_price المفقود أو الصفري المستبعدة من الخصم': int((~valid_base_price).sum()),
})
display(commercial_kpis.to_frame('القيمة'))
assert 0 <= any_promotion_rate_pct <= 100
assert int(df['num_orders'].sum()) == num_orders_sum

وعد فؤاد: حساب متوسط الطلب الأسبوعي والانحراف المعياري ومعامل التذبذب لكل وجبة.


In [6]:
meal_weekly_orders = (
    df.groupby(['meal_id', 'week'])['num_orders']
    .sum()
    .rename('إجمالي طلبات الوجبة في الأسبوع عبر المراكز')
    .reset_index()
)
meal_volatility = (
    meal_weekly_orders.groupby('meal_id')['إجمالي طلبات الوجبة في الأسبوع عبر المراكز']
    .agg(متوسط_الطلب_الأسبوعي='mean', انحراف_معياري_للطلب_الأسبوعي='std', عدد_الأسابيع_المسجلة='count')
)
meal_volatility['معامل التذبذب'] = meal_volatility['انحراف_معياري_للطلب_الأسبوعي'] / meal_volatility['متوسط_الطلب_الأسبوعي']
meal_volatility['معامل التذبذب (%)'] = meal_volatility['معامل التذبذب'] * 100
meal_volatility = meal_volatility.sort_values('معامل التذبذب', ascending=False)
display(meal_volatility)
if (meal_volatility['متوسط_الطلب_الأسبوعي'] == 0).any():
    print('تنبيه: معامل التذبذب غير معرّف للوجبات ذات متوسط الطلب الصفري.')
print('التعريف: الانحراف المعياري للعينة لطلبات الوجبة الأسبوعية المرصودة عبر المراكز، مقسومًا على متوسطها (ddof=1). الأسابيع غير المسجلة لم تُملأ بصفر.')

,متوسط_الطلب_الأسبوعي,انحراف_معياري_للطلب_الأسبوعي,عدد_الأسابيع_المسجلة,معامل التذبذب,معامل التذبذب (%)
meal_id,,,,,
2444,6500.468966,7833.130305,145,1.205010,120.501003
1971,39507.724138,43605.509263,145,1.103721,110.372111
1445,7304.972414,7986.194417,145,1.093255,109.325456
2867,4923.620690,4953.652809,145,1.006100,100.609960
2490,29673.000000,27959.324242,125,0.942248,94.224798
2290,67381.400000,60641.916250,145,0.899980,89.998006
1754,47021.096552,41698.021104,145,0.886794,88.679389
2104,1838.586207,1513.660184,87,0.823274,82.327398
1727,32661.979310,26274.639486,145,0.804441,80.444113


التعريف: الانحراف المعياري للعينة لطلبات الوجبة الأسبوعية المرصودة عبر المراكز، مقسومًا على متوسطها (ddof=1). الأسابيع غير المسجلة لم تُملأ بصفر.


وعد فؤاد: التحقق من اتساق أعداد الصفوف ومجاميع الطلب بين ملخصات التحليل.


In [7]:
# تحقق نهائي للمجاميع، مع الإبقاء على جميع الصفوف كما قُرئت.
assert len(df) == quality_summary['الصفوف']
assert int(df['num_orders'].sum()) == int(weekly_orders.sum()) == num_orders_sum
assert int(center_totals['إجمالي الطلبات المرصودة'].sum()) == num_orders_sum
print('نجح التحقق النهائي: أعداد الصفوف ومجاميع الطلب متسقة بين البيانات والتجميع الأسبوعي وتجميع المراكز.')
print(f'النطاق المرصود: الأسابيع {week_min}–{week_max}. لم تُحذف قيم شاذة ولم تُعدّل البيانات الأصلية.')
print('القيود: الأسابيع ليست تواريخ تقويمية، والمدن ممثلة برموز، والملف لا يتضمن بيانات المخزون أو الهدر أو هامش الربح. النتائج وصفية ولا تثبت السببية.')

نجح التحقق النهائي: أعداد الصفوف ومجاميع الطلب متسقة بين البيانات والتجميع الأسبوعي وتجميع المراكز.
النطاق المرصود: الأسابيع 1–145. لم تُحذف قيم شاذة ولم تُعدّل البيانات الأصلية.
القيود: الأسابيع ليست تواريخ تقويمية، والمدن ممثلة برموز، والملف لا يتضمن بيانات المخزون أو الهدر أو هامش الربح. النتائج وصفية ولا تثبت السببية.


## مقارنة الطلب حسب الترويج والخصم

تُحلّل مؤشرات البريد والصفحة الرئيسية كلٌّ على حدة. تقتصر المقارنة على أزواج الوجبة والمركز التي ظهر فيها المؤشر بحالتيه (0 و1)، ثم يُحسب فرق المتوسط النسبي داخل كل زوج ويُبلّغ متوسط هذه الفروق. ويُجرى Mann–Whitney على مشاهدات تلك الأزواج المشتركة، مع إظهار حجم كل مجموعة. قيم p لا تعالج الاعتماد الزمني بين الأسابيع؛ لذلك تُفسّر بحذر، وتُعرض أيضًا قيمة مصححة للاختبارين باستخدام Bonferroni. لا يدل الاختبار الرصدي على السببية.

وعد فؤاد: إجراء اختبار Mann–Whitney للطلب مع الترويج وبدونه، واختبار Spearman للخصم والطلب.


In [8]:
from scipy.stats import mannwhitneyu, norm, spearmanr, t as student_t

promotion_columns = ['emailer_for_promotion', 'homepage_featured']
for column in promotion_columns:
    observed_values = set(df[column].dropna().unique())
    if not observed_values.issubset({0, 1}):
        raise ValueError(f'قيم غير متوقعة في {column}: {sorted(observed_values)}')
    if df[column].isna().any():
        raise ValueError(f'قيم مفقودة في {column}; لا تُسقط بصمت.')

promotion_results = []
pair_columns = ['meal_id', 'center_id']
for promotion_column in promotion_columns:
    pair_states = df.groupby(pair_columns)[promotion_column].agg(['min', 'max'])
    eligible_pairs = pair_states.index[(pair_states['min'] == 0) & (pair_states['max'] == 1)]
    row_pairs = pd.MultiIndex.from_frame(df[pair_columns])
    matched_rows = df.loc[row_pairs.isin(eligible_pairs), pair_columns + ['num_orders', promotion_column]].copy()
    group_off = matched_rows.loc[matched_rows[promotion_column] == 0, 'num_orders']
    group_on = matched_rows.loc[matched_rows[promotion_column] == 1, 'num_orders']
    if group_off.empty or group_on.empty:
        raise ValueError(f'لا توجد عينات كافية لمقارنة {promotion_column}.')

    pair_means = matched_rows.groupby(pair_columns + [promotion_column])['num_orders'].mean().unstack(promotion_column)
    pair_means = pair_means.dropna(subset=[0, 1])
    if (pair_means[0] <= 0).any():
        raise ValueError(f'متوسط الطلب دون الترويج غير موجب في {promotion_column}; نسبة الفرق غير معرفة.')
    pair_pct_differences = (pair_means[1] - pair_means[0]) / pair_means[0] * 100

    test = mannwhitneyu(group_on, group_off, alternative='two-sided', method='asymptotic')
    log_p_value = float(norm.logsf(abs(test.zstatistic)) + np.log(2))
    log10_p_value = log_p_value / np.log(10)
    promotion_results.append({
        'مؤشر الترويج': promotion_column,
        'أزواج الوجبة-المركز ذات الحالتين': len(eligible_pairs),
        'عدد المشاهدات دون الترويج': len(group_off),
        'عدد المشاهدات مع الترويج': len(group_on),
        'متوسط الطلب دون الترويج': float(group_off.mean()),
        'وسيط الطلب دون الترويج': float(group_off.median()),
        'متوسط الطلب مع الترويج': float(group_on.mean()),
        'وسيط الطلب مع الترويج': float(group_on.median()),
        'متوسط فرق المتوسط داخل الزوج (%)': float(pair_pct_differences.mean()),
        'وسيط فرق المتوسط داخل الزوج (%)': float(pair_pct_differences.median()),
        'Mann–Whitney U': float(test.statistic),
        'قيمة p غير المصححة': float(test.pvalue),
        'log10 لقيمة p غير المصححة': float(log10_p_value),
    })

promotion_summary = pd.DataFrame(promotion_results).set_index('مؤشر الترويج')
promotion_summary['قيمة p مصححة Bonferroni لاختبارين'] = (promotion_summary['قيمة p غير المصححة'] * len(promotion_columns)).clip(upper=1.0)
promotion_summary['log10 لقيمة p المصححة'] = (promotion_summary['log10 لقيمة p غير المصححة'] + np.log10(len(promotion_columns))).clip(upper=0.0)
def format_p_value(value, log10_value=None):
    if value == 0:
        if log10_value is not None and np.isfinite(log10_value):
            return f'< 1e-300 (log10 p ≈ {log10_value:.1f})'
        return '< 1e-300 (numerical underflow)'
    return f'{value:.3g}'

promotion_display = promotion_summary.copy()
promotion_display['قيمة p غير المصححة'] = [format_p_value(row['قيمة p غير المصححة'], row['log10 لقيمة p غير المصححة']) for _, row in promotion_summary.iterrows()]
promotion_display['قيمة p مصححة Bonferroni لاختبارين'] = [format_p_value(row['قيمة p مصححة Bonferroni لاختبارين'], row['log10 لقيمة p المصححة']) for _, row in promotion_summary.iterrows()]
display(promotion_display)
print('فرق النسبة داخل زوج الوجبة-المركز = (متوسط الطلب عند الترويج - متوسطه بدونه) ÷ متوسطه بدونه × 100؛ متوسط الفروق يعطي كل زوج وزنًا متساويًا.')
print('اختبار Mann–Whitney مطبّق على مشاهدات الأزواج ذات الحالتين مجتمعة؛ لا يضبط الاعتماد بين الأسابيع المتكررة، لذا فقيمة p وصفية وليست دليلًا سببيًا.')

,أزواج الوجبة-المركز ذات الحالتين,عدد المشاهدات دون الترويج,عدد المشاهدات مع الترويج,متوسط الطلب دون الترويج,وسيط الطلب دون الترويج,متوسط الطلب مع الترويج,وسيط الطلب مع الترويج,متوسط فرق المتوسط داخل الزوج (%),وسيط فرق المتوسط داخل الزوج (%),Mann–Whitney U,قيمة p غير المصححة,log10 لقيمة p غير المصححة,قيمة p مصححة Bonferroni لاختبارين,log10 لقيمة p المصححة
مؤشر الترويج,,,,,,,,,,,,,,
emailer_for_promotion,2830,338694,37050,254.642539,149.0,631.097544,338.0,167.093473,142.139384,8.740582e+09,< 1e-300 (log10 p ≈ -3364.1),-3364.137981,< 1e-300 (log10 p ≈ -3363.8),-3363.836951
homepage_featured,2928,338548,49855,241.372937,148.0,594.884786,350.0,149.114734,122.307181,1.169418e+10,< 1e-300 (log10 p ≈ -4214.3),-4214.304255,< 1e-300 (log10 p ≈ -4214.0),-4214.003225


فرق النسبة داخل زوج الوجبة-المركز = (متوسط الطلب عند الترويج - متوسطه بدونه) ÷ متوسطه بدونه × 100؛ متوسط الفروق يعطي كل زوج وزنًا متساويًا.
اختبار Mann–Whitney مطبّق على مشاهدات الأزواج ذات الحالتين مجتمعة؛ لا يضبط الاعتماد بين الأسابيع المتكررة، لذا فقيمة p وصفية وليست دليلًا سببيًا.


وعد فؤاد: تحديد الخصوم الصالحة وحساب علاقتها بالطلب إحصائيًا مع توثيق حجم العينة.


In [9]:
valid_discount = (
    df['base_price'].notna()
    & np.isfinite(df['base_price'])
    & (df['base_price'] != 0)
    & df['checkout_price'].notna()
    & np.isfinite(df['checkout_price'])
    & df['num_orders'].notna()
    & np.isfinite(df['num_orders'])
)
discount_analysis = df.loc[valid_discount, ['base_price', 'checkout_price', 'num_orders']].copy()
discount_analysis['discount_rate'] = (discount_analysis['base_price'] - discount_analysis['checkout_price']) / discount_analysis['base_price']
if len(discount_analysis) < 2:
    raise ValueError('لا توجد مشاهدات كافية لحساب ارتباط Spearman.')
spearman_result = spearmanr(discount_analysis['discount_rate'], discount_analysis['num_orders'])
spearman_t_statistic = spearman_result.statistic * np.sqrt((len(discount_analysis) - 2) / ((1 + spearman_result.statistic) * (1 - spearman_result.statistic)))
spearman_log10_p = float((np.log(2) + student_t.logsf(abs(spearman_t_statistic), len(discount_analysis) - 2)) / np.log(10))
spearman_summary = pd.Series({
    'عدد المشاهدات المستخدمة': len(discount_analysis),
    'الصفوف المستبعدة لأسعار غير صالحة': len(df) - len(discount_analysis),
    'Spearman rho': float(spearman_result.statistic),
    'قيمة p': float(spearman_result.pvalue),
    'اتجاه الارتباط': 'موجب' if spearman_result.statistic > 0 else ('سالب' if spearman_result.statistic < 0 else 'لا ارتباط رتيب مرصود'),
})
spearman_display = spearman_summary.copy()
spearman_display['قيمة p'] = format_p_value(spearman_result.pvalue, spearman_log10_p)
display(spearman_display.to_frame('القيمة'))
print('نسبة الخصم = (base_price - checkout_price) ÷ base_price للأسعار الأساسية غير الصفرية والصفوف ذات القيم العددية المنتهية.')
print('ارتباط الخصم بالطلب لا يثبت السببية؛ وقد يتزامن الخصم مع البريد الترويجي أو الظهور في الصفحة الرئيسية.')

,القيمة
عدد المشاهدات المستخدمة,456548
الصفوف المستبعدة لأسعار غير صالحة,0
Spearman rho,0.121813
قيمة p,< 1e-300 (numerical underflow)
اتجاه الارتباط,موجب


نسبة الخصم = (base_price - checkout_price) ÷ base_price للأسعار الأساسية غير الصفرية والصفوف ذات القيم العددية المنتهية.
ارتباط الخصم بالطلب لا يثبت السببية؛ وقد يتزامن الخصم مع البريد الترويجي أو الظهور في الصفحة الرئيسية.


## ترتيب الوجبات بحسب حجم الطلب والتذبذب

يجمع الترتيب بين رتبة إجمالي الطلب المرصود عبر الأسابيع والمراكز ورتبة معامل التذبذب المحسوب سابقًا للطلبات الأسبوعية المجمعة عبر المراكز. متوسط الرتبتين يوازن الحجم والتذبذب، وتُعرض المكونات منفصلة حتى لا يخفي الترتيب المركب اختلافهما.

وعد فؤاد: ترتيب الوجبات بحسب حجم الطلب وتذبذبه لتحديد أولويات متابعة التخطيط.


In [10]:
meal_total_orders = df.groupby('meal_id')['num_orders'].sum().rename('إجمالي الطلبات')
meal_priority = meal_total_orders.to_frame().join(
    meal_volatility[['متوسط_الطلب_الأسبوعي', 'معامل التذبذب', 'معامل التذبذب (%)']]
)
meal_priority['رتبة حجم الطلب'] = meal_priority['إجمالي الطلبات'].rank(method='min', ascending=False)
meal_priority['رتبة التذبذب'] = meal_priority['معامل التذبذب'].rank(method='min', ascending=False)
meal_priority['متوسط الرتبتين'] = (meal_priority['رتبة حجم الطلب'] + meal_priority['رتبة التذبذب']) / 2
meal_priority = meal_priority.sort_values(
    ['متوسط الرتبتين', 'إجمالي الطلبات'], ascending=[True, False]
)
display(meal_priority.head(20))
print('تنبيه: الرتب وسيلة فرز وصفية متساوية الوزن، وليست توصية مخزون؛ لا تتوفر بيانات المخزون أو الهدر أو مهلة التوريد.')

,إجمالي الطلبات,متوسط_الطلب_الأسبوعي,معامل التذبذب,معامل التذبذب (%),رتبة حجم الطلب,رتبة التذبذب,متوسط الرتبتين
meal_id,,,,,,,
2290,9770303,67381.400000,0.899980,89.998006,1.0,6.0,3.5
1971,5728620,39507.724138,1.103721,110.372111,7.0,2.0,4.5
1754,6818059,47021.096552,0.886794,88.679389,3.0,7.0,5.0
2490,3709125,29673.000000,0.942248,94.224798,12.0,5.0,8.5
1727,4735987,32661.979310,0.804441,80.444113,9.0,9.0,9.0
1109,6367773,43915.675862,0.560624,56.062441,5.0,16.0,10.5
1962,3235780,22315.724138,0.684183,68.418271,15.0,13.0,14.0
2581,2399000,16544.827586,0.721328,72.132822,17.0,12.0,14.5
2826,5090103,35104.158621,0.485955,48.595490,8.0,24.0,16.0


تنبيه: الرتب وسيلة فرز وصفية متساوية الوزن، وليست توصية مخزون؛ لا تتوفر بيانات المخزون أو الهدر أو مهلة التوريد.


## استقرار الطلب ومخاطر التخطيط

يُلخّص هذا القسم الطلب الأسبوعي لكل وجبة بعد جمعه عبر المراكز، ثم يحسب إجمالي الطلب المرصود، ومتوسط الطلب والانحراف المعياري للعينة وCV. تُستخدم وسيطات الوجبات المرصودة حدودًا وصفية نسبية: القيمة عند الوسيط أو فوقه تصنّف مرتفعة نسبيًا، وما دونه منخفضة نسبيًا. لا تُملأ أسابيع الوجبة غير المسجلة بصفر؛ لذلك تُعرض أعداد أسابيعها المسجلة. هذه الشرائح لا تمثل حدود إعادة طلب ولا إثباتًا لنفاد المخزون أو الهدر.

وعد فؤاد: تصنيف الوجبات حسب مستوى الطلب والاستقرار باستخدام حدود وسيط واضحة.


In [11]:
meal_stability = (
    meal_weekly_orders.groupby('meal_id')['إجمالي طلبات الوجبة في الأسبوع عبر المراكز']
    .agg(
        إجمالي_الطلب_المرصود='sum',
        متوسط_الطلب_الأسبوعي='mean',
        الانحراف_المعياري_الأسبوعي='std',
        الأسابيع_المسجلة='count',
    )
)
meal_stability['معامل التذبذب CV'] = (
    meal_stability['الانحراف_المعياري_الأسبوعي']
    / meal_stability['متوسط_الطلب_الأسبوعي']
)
if meal_stability['معامل التذبذب CV'].isna().any() or not np.isfinite(meal_stability['معامل التذبذب CV']).all():
    raise ValueError('معامل التذبذب غير معرّف أو غير منتهٍ لوجبة واحدة على الأقل.')

demand_threshold = float(meal_stability['متوسط_الطلب_الأسبوعي'].median())
volatility_threshold = float(meal_stability['معامل التذبذب CV'].median())
meal_stability['فئة الطلب'] = np.where(
    meal_stability['متوسط_الطلب_الأسبوعي'] >= demand_threshold,
    'مرتفع نسبيًا',
    'منخفض نسبيًا',
)
meal_stability['فئة التذبذب'] = np.where(
    meal_stability['معامل التذبذب CV'] >= volatility_threshold,
    'مرتفع نسبيًا',
    'منخفض نسبيًا',
)
meal_stability['مجموعة التخطيط الوصفية'] = np.select(
    [
        (meal_stability['فئة الطلب'] == 'مرتفع نسبيًا') & (meal_stability['فئة التذبذب'] == 'مرتفع نسبيًا'),
        (meal_stability['فئة الطلب'] == 'مرتفع نسبيًا') & (meal_stability['فئة التذبذب'] == 'منخفض نسبيًا'),
    ],
    ['طلب أعلى نسبيًا وتذبذب أعلى نسبيًا', 'طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا'],
    default='طلب أقل نسبيًا',
)

assert len(meal_stability) == df['meal_id'].nunique()
assert int(meal_stability['إجمالي_الطلب_المرصود'].sum()) == num_orders_sum
assert int(meal_stability['الأسابيع_المسجلة'].sum()) == len(meal_weekly_orders)
assert meal_stability['الأسابيع_المسجلة'].between(1, len(weekly_orders)).all()

planning_candidates = meal_stability.loc[
    meal_stability['فئة الطلب'] == 'مرتفع نسبيًا'
].sort_values(['فئة التذبذب', 'معامل التذبذب CV'], ascending=[True, False])
high_demand_high_variability = planning_candidates.loc[
    planning_candidates['فئة التذبذب'] == 'مرتفع نسبيًا'
]
high_demand_more_stable = planning_candidates.loc[
    planning_candidates['فئة التذبذب'] == 'منخفض نسبيًا'
]

print(f'حد الطلب المرتفع نسبيًا: متوسط أسبوعي >= وسيط الوجبات = {demand_threshold:,.2f} طلبًا/أسبوعًا')
print(f'حد التذبذب المرتفع نسبيًا: CV >= وسيط الوجبات = {volatility_threshold:.4f} ({volatility_threshold * 100:.2f}%)')
print(f'عدد الوجبات: {len(meal_stability)}؛ طلب مرتفع وتذبذب مرتفع: {len(high_demand_high_variability)}؛ طلب مرتفع وأكثر استقرارًا نسبيًا: {len(high_demand_more_stable)}')

print('مرشحات لمراجعة مخاطر التخطيط (طلب مرتفع وتذبذب مرتفع نسبيًا؛ ليست إثباتًا لمخاطر مخزون فعلية):')
display(high_demand_high_variability)
print('مرشحات لتخطيط أكثر قابلية للتوقع نسبيًا (طلب مرتفع وتذبذب منخفض نسبيًا):')
display(high_demand_more_stable)
print('الجدول الكامل وتصنيف كل وجبة:')
display(meal_stability.sort_values(['مجموعة التخطيط الوصفية', 'إجمالي_الطلب_المرصود'], ascending=[True, False]))
print('التوصية التشغيلية: استخدم هذه الشرائح لترتيب مراجعة الطلب، ولا تحدد مخزونًا احتياطيًا قبل توفر بيانات المخزون والهدر ومهلة التوريد.')

حد الطلب المرتفع نسبيًا: متوسط أسبوعي >= وسيط الوجبات = 10,273.66 طلبًا/أسبوعًا
حد التذبذب المرتفع نسبيًا: CV >= وسيط الوجبات = 0.4598 (45.98%)
عدد الوجبات: 51؛ طلب مرتفع وتذبذب مرتفع: 14؛ طلب مرتفع وأكثر استقرارًا نسبيًا: 12
مرشحات لمراجعة مخاطر التخطيط (طلب مرتفع وتذبذب مرتفع نسبيًا؛ ليست إثباتًا لمخاطر مخزون فعلية):


,إجمالي_الطلب_المرصود,متوسط_الطلب_الأسبوعي,الانحراف_المعياري_الأسبوعي,الأسابيع_المسجلة,معامل التذبذب CV,فئة الطلب,فئة التذبذب,مجموعة التخطيط الوصفية
meal_id,,,,,,,,
1971,5728620,39507.724138,43605.509263,145,1.103721,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
2490,3709125,29673.000000,27959.324242,125,0.942248,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
2290,9770303,67381.400000,60641.916250,145,0.899980,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
1754,6818059,47021.096552,41698.021104,145,0.886794,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
1727,4735987,32661.979310,26274.639486,145,0.804441,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
1558,1748940,12061.655172,9383.111367,145,0.777929,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
2581,2399000,16544.827586,11934.251022,145,0.721328,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
1962,3235780,22315.724138,15268.032510,145,0.684183,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا
1109,6367773,43915.675862,24620.199675,145,0.560624,مرتفع نسبيًا,مرتفع نسبيًا,طلب أعلى نسبيًا وتذبذب أعلى نسبيًا


مرشحات لتخطيط أكثر قابلية للتوقع نسبيًا (طلب مرتفع وتذبذب منخفض نسبيًا):


,إجمالي_الطلب_المرصود,متوسط_الطلب_الأسبوعي,الانحراف_المعياري_الأسبوعي,الأسابيع_المسجلة,معامل التذبذب CV,فئة الطلب,فئة التذبذب,مجموعة التخطيط الوصفية
meal_id,,,,,,,,
2664,3900812,29329.413534,12178.818057,133,0.415242,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1878,1542027,10634.668966,4303.036501,145,0.404623,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2640,2085360,14381.793103,5654.274180,145,0.393155,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2704,1610896,11109.627586,4246.174348,145,0.382207,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1525,1719378,11857.779310,4260.409765,145,0.359292,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1778,3308383,22816.434483,5856.903989,145,0.256697,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2539,3234528,22307.089655,5348.400908,145,0.239762,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2707,5932792,40915.806897,9760.738748,145,0.238557,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1993,6656856,45909.351724,10707.705831,145,0.233236,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا


الجدول الكامل وتصنيف كل وجبة:


,إجمالي_الطلب_المرصود,متوسط_الطلب_الأسبوعي,الانحراف_المعياري_الأسبوعي,الأسابيع_المسجلة,معامل التذبذب CV,فئة الطلب,فئة التذبذب,مجموعة التخطيط الوصفية
meal_id,,,,,,,,
1885,8346246,57560.317241,10998.041724,145,0.191070,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1993,6656856,45909.351724,10707.705831,145,0.233236,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2707,5932792,40915.806897,9760.738748,145,0.238557,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1062,4712795,32502.034483,6382.437463,145,0.196370,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2664,3900812,29329.413534,12178.818057,133,0.415242,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1778,3308383,22816.434483,5856.903989,145,0.256697,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2539,3234528,22307.089655,5348.400908,145,0.239762,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
2640,2085360,14381.793103,5654.274180,145,0.393155,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا
1525,1719378,11857.779310,4260.409765,145,0.359292,مرتفع نسبيًا,منخفض نسبيًا,طلب أعلى نسبيًا وأكثر استقرارًا نسبيًا


التوصية التشغيلية: استخدم هذه الشرائح لترتيب مراجعة الطلب، ولا تحدد مخزونًا احتياطيًا قبل توفر بيانات المخزون والهدر ومهلة التوريد.


## فرص التسعير والترويج

يستخدم هذا القسم خصم الصف بوصفه `(base_price - checkout_price) / base_price`، ومتوسط الخصم لكل وجبة عبر الصفوف ذات الأسعار الصالحة. يُعرّف الطلب لكل وجبة بمتوسط إجمالي طلبها في الأسابيع المسجلة، لا بمجموعها الخام، لتقليل أثر اختلاف عدد الأسابيع المرصودة. حدود «مرتفع/منخفض» هي وسيط المؤشر عبر الوجبات الـ51؛ المساواة مع الوسيط تدخل في فئة «مرتفع». هذه حدود مقارنة داخل العينة وليست عتبات تجارية معيارية.

يقارن جدول مستويات الخصم الطلب على مستوى صفوف البيانات، مع عدد الصفوف الفعلي بكل مجموعة. جدول فرص المراجعة يصنف الوجبات على مستوى المنتج ويعرض عدد صفوف الأسعار الصالحة لكل وجبة. يعاد استخدام نتيجة Spearman المحسوبة أعلاه بدل إجراء الاختبار مرة أخرى.

وعد فؤاد: تصنيف مستويات الطلب والخصم لكل وجبة وعرض مجموعات تستحق مراجعة تجارية.


In [12]:
if 'discount_analysis' not in globals() or 'spearman_result' not in globals():
    raise RuntimeError('يلزم تشغيل خلية Spearman السابقة أولًا؛ لا يُكرر الاختبار في هذا القسم.')

discount_row_threshold = float(discount_analysis['discount_rate'].median())
discount_by_row = discount_analysis[['discount_rate', 'num_orders']].copy()
discount_by_row['مستوى الخصم'] = np.where(
    discount_by_row['discount_rate'] >= discount_row_threshold,
    'مرتفع نسبيًا (عند الوسيط أو فوقه)',
    'منخفض نسبيًا (دون الوسيط)',
)
discount_level_summary = (
    discount_by_row.groupby('مستوى الخصم', sort=False)
    .agg(
        حجم_العينة_صفوف=('num_orders', 'size'),
        متوسط_الطلب=('num_orders', 'mean'),
        وسيط_الطلب=('num_orders', 'median'),
        إجمالي_الطلبات=('num_orders', 'sum'),
        متوسط_الخصم=('discount_rate', 'mean'),
    )
)
discount_level_summary['متوسط الخصم (%)'] = discount_level_summary.pop('متوسط_الخصم') * 100
display(discount_level_summary)
print(f'حد الخصم على مستوى الصفوف: وسيط الخصم الصالح = {discount_row_threshold * 100:.4f}%. العينة هنا صفوف وجبة-مركز-أسبوع وليست عدد وجبات.')
markup_rows = int((discount_analysis['checkout_price'] > discount_analysis['base_price']).sum())
print(f'صفوف سعر البيع الأعلى من السعر الأساسي: {markup_rows:,}. لم تُستبعد؛ لذلك قد تكون نسب الخصم سالبة، وهي فرق سعر محسوب وليست قيمة مفقودة أو صفًا مستبعدًا.')

if not np.isfinite(spearman_result.statistic):
    raise ValueError('نتيجة Spearman السابقة غير منتهية؛ لا يمكن اعتمادها.')
spearman_reused = pd.Series({
    'حجم عينة الصفوف': len(discount_analysis),
    'Spearman rho': float(spearman_result.statistic),
    'قيمة p (محسوبة في قسم العروض والخصم)': format_p_value(float(spearman_result.pvalue), spearman_log10_p),
    'الصفوف المستبعدة لخصم غير صالح': len(df) - len(discount_analysis),
})
display(spearman_reused.to_frame('القيمة'))

discount_rows_by_meal = discount_analysis.join(df['meal_id'])
meal_discount_summary = (
    discount_rows_by_meal.groupby('meal_id')['discount_rate']
    .agg(متوسط_الخصم='mean', صفوف_الأسعار_الصالحة='size')
)
meal_weekly_for_pricing = meal_weekly_orders.groupby('meal_id')[
    'إجمالي طلبات الوجبة في الأسبوع عبر المراكز'
].agg(
    إجمالي_الطلب_المرصود='sum',
    متوسط_الطلب_الأسبوعي_لأسابيع_الظهور='mean',
    الأسابيع_المرصودة='count',
)
meal_pricing = meal_weekly_for_pricing.join(meal_discount_summary, how='inner')
if len(meal_pricing) != df['meal_id'].nunique():
    raise ValueError('تغطية الوجبات تختلف بين الطلب والأسعار؛ راجع المفاتيح قبل المقارنة.')
demand_meal_threshold = float(meal_pricing['متوسط_الطلب_الأسبوعي_لأسابيع_الظهور'].median())
discount_meal_threshold = float(meal_pricing['متوسط_الخصم'].median())
meal_pricing['مستوى الطلب'] = np.where(
    meal_pricing['متوسط_الطلب_الأسبوعي_لأسابيع_الظهور'] >= demand_meal_threshold,
    'طلب مرتفع نسبيًا', 'طلب منخفض نسبيًا',
)
meal_pricing['مستوى الخصم'] = np.where(
    meal_pricing['متوسط_الخصم'] >= discount_meal_threshold,
    'خصم مرتفع نسبيًا', 'خصم منخفض نسبيًا',
)
meal_pricing['مجموعة الطلب والخصم'] = meal_pricing['مستوى الطلب'] + ' / ' + meal_pricing['مستوى الخصم']

pricing_segments = (
    meal_pricing.groupby('مجموعة الطلب والخصم', sort=True)
    .agg(
        عدد_الوجبات=('إجمالي_الطلب_المرصود', 'size'),
        صفوف_الأسعار_الصالحة=('صفوف_الأسعار_الصالحة', 'sum'),
        إجمالي_الطلبات=('إجمالي_الطلب_المرصود', 'sum'),
        متوسط_الطلب_الأسبوعي_للفئة=('متوسط_الطلب_الأسبوعي_لأسابيع_الظهور', 'mean'),
        متوسط_الخصم_للفئة=('متوسط_الخصم', 'mean'),
    )
)
pricing_segments['متوسط الخصم للفئة (%)'] = pricing_segments.pop('متوسط_الخصم_للفئة') * 100
display(pricing_segments)

high_demand_low_discount = meal_pricing.loc[
    (meal_pricing['مستوى الطلب'] == 'طلب مرتفع نسبيًا')
    & (meal_pricing['مستوى الخصم'] == 'خصم منخفض نسبيًا')
].sort_values('متوسط_الطلب_الأسبوعي_لأسابيع_الظهور', ascending=False)
high_discount_low_demand = meal_pricing.loc[
    (meal_pricing['مستوى الطلب'] == 'طلب منخفض نسبيًا')
    & (meal_pricing['مستوى الخصم'] == 'خصم مرتفع نسبيًا')
].sort_values(['متوسط_الخصم', 'متوسط_الطلب_الأسبوعي_لأسابيع_الظهور'], ascending=[False, True])

candidate_columns = [
    'إجمالي_الطلب_المرصود', 'متوسط_الطلب_الأسبوعي_لأسابيع_الظهور',
    'الأسابيع_المرصودة', 'صفوف_الأسعار_الصالحة', 'متوسط_الخصم',
]
print(f'حد الطلب لكل وجبة: وسيط متوسط الطلب في أسابيع الظهور = {demand_meal_threshold:,.2f} طلبًا/أسبوعًا عبر المراكز.')
print(f'حد الخصم لكل وجبة: وسيط متوسط الخصم = {discount_meal_threshold * 100:.2f}%.')
print('نتيجة وصفية تستحق مراجعة تجارية: طلب مرتفع نسبيًا وخصم منخفض نسبيًا؛ هذا لا يثبت أن الخصم غير ضروري أو أن رفع السعر لن يغير الطلب.')
display(high_demand_low_discount[candidate_columns].assign(متوسط_الخصم_بالمئة=lambda t: t['متوسط_الخصم'] * 100).drop(columns='متوسط_الخصم'))
print('نتيجة وصفية تستحق مراجعة تجارية: خصم مرتفع نسبيًا وطلب منخفض نسبيًا؛ هذا لا يثبت أن الخصم سبب انخفاض الطلب أو أنه غير فعال.')
display(high_discount_low_demand[candidate_columns].assign(متوسط_الخصم_بالمئة=lambda t: t['متوسط_الخصم'] * 100).drop(columns='متوسط_الخصم'))

assert int(meal_pricing['إجمالي_الطلب_المرصود'].sum()) == num_orders_sum
assert int(meal_pricing['صفوف_الأسعار_الصالحة'].sum()) == len(discount_analysis)
assert int(pricing_segments['عدد_الوجبات'].sum()) == len(meal_pricing)
assert int(discount_level_summary['حجم_العينة_صفوف'].sum()) == len(discount_analysis)
print('نجح التحقق: مجاميع الطلب وعدد الوجبات وصفوف الأسعار متسقة مع البيانات المستخدمة.')

,حجم_العينة_صفوف,متوسط_الطلب,وسيط_الطلب,إجمالي_الطلبات,متوسط الخصم (%)
مستوى الخصم,,,,,
مرتفع نسبيًا (عند الوسيط أو فوقه),228303,306.359089,161.0,69942699,11.789005
منخفض نسبيًا (دون الوسيط),228245,217.375128,122.0,49614786,-0.351547


حد الخصم على مستوى الصفوف: وسيط الخصم الصالح = 0.1412%. العينة هنا صفوف وجبة-مركز-أسبوع وليست عدد وجبات.
صفوف سعر البيع الأعلى من السعر الأساسي: 116,101. لم تُستبعد؛ لذلك قد تكون نسب الخصم سالبة، وهي فرق سعر محسوب وليست قيمة مفقودة أو صفًا مستبعدًا.


,القيمة
حجم عينة الصفوف,456548
Spearman rho,0.121813
قيمة p (محسوبة في قسم العروض والخصم),< 1e-300 (numerical underflow)
الصفوف المستبعدة لخصم غير صالح,0


,عدد_الوجبات,صفوف_الأسعار_الصالحة,إجمالي_الطلبات,متوسط_الطلب_الأسبوعي_للفئة,متوسط الخصم للفئة (%)
مجموعة الطلب والخصم,,,,,
طلب مرتفع نسبيًا / خصم مرتفع نسبيًا,12,121489,46441524,27031.600000,7.426619
طلب مرتفع نسبيًا / خصم منخفض نسبيًا,14,150627,56979365,28445.757059,3.037434
طلب منخفض نسبيًا / خصم مرتفع نسبيًا,14,123390,12190411,6005.128571,9.405481
طلب منخفض نسبيًا / خصم منخفض نسبيًا,11,61042,3946185,2838.552351,2.777405


حد الطلب لكل وجبة: وسيط متوسط الطلب في أسابيع الظهور = 10,273.66 طلبًا/أسبوعًا عبر المراكز.
حد الخصم لكل وجبة: وسيط متوسط الخصم = 4.56%.
نتيجة وصفية تستحق مراجعة تجارية: طلب مرتفع نسبيًا وخصم منخفض نسبيًا؛ هذا لا يثبت أن الخصم غير ضروري أو أن رفع السعر لن يغير الطلب.


,إجمالي_الطلب_المرصود,متوسط_الطلب_الأسبوعي_لأسابيع_الظهور,الأسابيع_المرصودة,صفوف_الأسعار_الصالحة,متوسط_الخصم_بالمئة
meal_id,,,,,
1885,8346246,57560.317241,145,11092,2.748104
1993,6656856,45909.351724,145,11116,2.590115
2707,5932792,40915.806897,145,11123,4.415686
2826,5090103,35104.158621,145,11057,3.616882
1727,4735987,32661.979310,145,11136,4.130266
1062,4712795,32502.034483,145,11137,2.512954
2664,3900812,29329.413534,133,9853,2.039556
2569,3334399,25848.054264,129,9459,2.329703
1778,3308383,22816.434483,145,11121,2.454478


نتيجة وصفية تستحق مراجعة تجارية: خصم مرتفع نسبيًا وطلب منخفض نسبيًا؛ هذا لا يثبت أن الخصم سبب انخفاض الطلب أو أنه غير فعال.


,إجمالي_الطلب_المرصود,متوسط_الطلب_الأسبوعي_لأسابيع_الظهور,الأسابيع_المرصودة,صفوف_الأسعار_الصالحة,متوسط_الخصم_بالمئة
meal_id,,,,,
2631,1090132,7518.151724,145,10458,16.191731
1248,874750,6032.758621,145,9939,15.974295
1198,1018280,7022.620690,145,4206,14.321498
1803,1477018,10186.331034,145,4674,13.354687
2306,506600,3493.793103,145,8294,12.883771
2867,713925,4923.620690,145,8092,8.149826
2444,942568,6500.468966,145,9097,8.133728
1445,1059221,7304.972414,145,9727,7.932835
2492,540339,3726.475862,145,9382,6.442840


نجح التحقق: مجاميع الطلب وعدد الوجبات وصفوف الأسعار متسقة مع البيانات المستخدمة.


وعد فؤاد: إنشاء الرسوم التحليلية وحفظها بصيغة PNG في مجلد التقارير.


In [13]:
import matplotlib.pyplot as plt

project_root = DATA_PATH.parent.parent.parent
figures_dir = project_root / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
plt.rcParams['font.family'] = ['Tahoma', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
colors = {
    'high_demand_low_discount': '#277DA1',
    'low_demand_high_discount': '#F9844A',
    'other': '#9AA0A6',
}

# 1) Meal-level demand versus average discount; thresholds are the medians across meals.
fig, ax = plt.subplots(figsize=(9, 6))
for _, row in meal_pricing.iterrows():
    if row['مستوى الطلب'] == 'طلب مرتفع نسبيًا' and row['مستوى الخصم'] == 'خصم منخفض نسبيًا':
        color, label = colors['high_demand_low_discount'], 'طلب مرتفع / خصم منخفض'
    elif row['مستوى الطلب'] == 'طلب منخفض نسبيًا' and row['مستوى الخصم'] == 'خصم مرتفع نسبيًا':
        color, label = colors['low_demand_high_discount'], 'طلب منخفض / خصم مرتفع'
    else:
        color, label = colors['other'], 'مجموعات أخرى'
    ax.scatter(row['متوسط_الخصم'] * 100, row['متوسط_الطلب_الأسبوعي_لأسابيع_الظهور'], color=color, alpha=0.85, s=48, label=label)
ax.axvline(discount_meal_threshold * 100, color='#333333', linestyle='--', linewidth=1)
ax.axhline(demand_meal_threshold, color='#333333', linestyle='--', linewidth=1)
ax.set_title('توزيع الوجبات حسب الطلب الأسبوعي والخصم المتوسط (51 وجبة)')
ax.set_xlabel('متوسط الخصم لكل وجبة (%)')
ax.set_ylabel('متوسط الطلب في أسابيع الظهور (طلب/أسبوع عبر المراكز)')
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), fontsize=8)
ax.grid(axis='both', alpha=0.2)
fig.tight_layout()
fig.savefig(figures_dir / 'pricing_demand_by_meal.png', dpi=160, bbox_inches='tight')
plt.close(fig)

# 2) Row-level order comparison by discount band; sample size is printed above each bar.
row_plot = discount_level_summary.reindex([
    'مرتفع نسبيًا (عند الوسيط أو فوقه)',
    'منخفض نسبيًا (دون الوسيط)',
]).copy()
row_plot.index = ['خصم عند الوسيط أو أعلى', 'خصم دون الوسيط']
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(row_plot.index, row_plot['متوسط_الطلب'], color=['#277DA1', '#8D99AE'])
for bar, (_, row) in zip(bars, row_plot.iterrows()):
    ax.annotate(f"n={int(row['حجم_العينة_صفوف']):,}\nالوسيط={row['وسيط_الطلب']:,.0f}",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha='center', va='bottom', fontsize=9, xytext=(0, 4), textcoords='offset points')
ax.set_ylim(top=float(row_plot['متوسط_الطلب'].max()) * 1.2)
ax.set_title('متوسط الطلب أعلى في مجموعة الخصم الأعلى؛ مقارنة وصفية على مستوى الصف')
ax.set_ylabel('متوسط الطلب لكل صف (وحدة طلب)')
ax.set_xlabel('مجموعة الخصم؛ الحد هو وسيط الصفوف الصالحة')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(figures_dir / 'row_demand_by_discount_band.png', dpi=160, bbox_inches='tight')
plt.close(fig)

# 3) Highest-demand meals inside the high-demand/low-discount review segment.
top_low_discount = high_demand_low_discount.nlargest(10, 'إجمالي_الطلب_المرصود').sort_values('إجمالي_الطلب_المرصود')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_low_discount.index.astype(str), top_low_discount['إجمالي_الطلب_المرصود'], color=colors['high_demand_low_discount'])
ax.set_title('أعلى الوجبات طلبًا ضمن شريحة الطلب المرتفع والخصم المنخفض')
ax.set_xlabel('إجمالي الطلبات المرصودة (كل الأسابيع والمراكز)')
ax.set_ylabel('معرّف الوجبة')
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(figures_dir / 'high_demand_low_discount_meals.png', dpi=160, bbox_inches='tight')
plt.close(fig)

# 4) Highest-discount meals inside the low-demand/high-discount review segment.
top_high_discount = high_discount_low_demand.nlargest(10, 'متوسط_الخصم').sort_values('متوسط_الخصم')
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_high_discount.index.astype(str), top_high_discount['متوسط_الخصم'] * 100, color=colors['low_demand_high_discount'])
ax.set_title('أعلى الخصومات المتوسطة ضمن شريحة الطلب المنخفض والخصم المرتفع')
ax.set_xlabel('متوسط الخصم لكل وجبة (%)')
ax.set_ylabel('معرّف الوجبة')
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(figures_dir / 'low_demand_high_discount_meals.png', dpi=160, bbox_inches='tight')
plt.close(fig)

saved_figures = sorted(figures_dir.glob('*.png'))
print('حُفظت رسوم التحليل في:', figures_dir)
for figure_path in saved_figures:
    print(figure_path.name, figure_path.stat().st_size, 'bytes')
assert len(saved_figures) >= 4, 'لم تُحفظ الرسوم الأربعة المطلوبة.'

حُفظت رسوم التحليل في: c:\Users\hp\Desktop\food-demand-analysis\reports\figures
demand_by_category.png 170914 bytes
discount_vs_demand.png 1391932 bytes
high_demand_low_discount_meals.png 42776 bytes
low_demand_high_discount_meals.png 41912 bytes
pricing_demand_by_meal.png 84056 bytes
promotion_vs_no_promotion.png 107115 bytes
row_demand_by_discount_band.png 47758 bytes
weekly_demand_trend.png 352447 bytes


## التنبؤ بالطلب والتحقق الزمني

يُحجز أحدث 10 أسابيع معنونة من `train.csv` للتحقق، ويُدرّب انحدار خطي على الأسابيع الأسبق فقط. تستخدم الميزات الأسبوع والمركز والوجبة والأسعار ومؤشري الترويج، بافتراض توفر معلومات السعر والترويج وقت إعداد التوقع. لا يستخدم معرّف الصف `id` كميزة. يقارن التقييم فقط صفوف الوجبة-المركز-الأسبوع التي لها طلب فعلي في الأسبوع السابق؛ ويُعامل خط الأساس كتوقع أسبوعي متجدد، بينما يظل النموذج مدرّبًا قبل بداية فترة التحقق. ولأن الطلب لا يكون سالبًا، تُحوّل أي توقعات سالبة إلى صفر قبل التقييم والاستخدام.

وعد فؤاد: قراءة بيانات التدريب الخام والتحقق من الأعمدة والمفاتيح، ثم تحديد أسابيع التدريب والتحقق زمنيًا.


In [14]:
from scipy.sparse import csr_matrix, hstack
from scipy.sparse.linalg import lsqr

forecast_train_path = project_root / 'data' / 'raw' / 'train.csv'
forecast_test_path = project_root / 'data' / 'raw' / 'test.csv'
if not forecast_train_path.is_file():
    raise FileNotFoundError(f'ملف التدريب غير موجود: {forecast_train_path}')

forecast_train = pd.read_csv(forecast_train_path)
forecast_required = {
    'week', 'center_id', 'meal_id', 'checkout_price', 'base_price',
    'emailer_for_promotion', 'homepage_featured', 'num_orders',
}
missing_forecast_columns = sorted(forecast_required - set(forecast_train.columns))
if missing_forecast_columns:
    raise ValueError(f'أعمدة التدريب المطلوبة غير موجودة: {missing_forecast_columns}')
if forecast_train[list(forecast_required)].isna().any().any():
    raise ValueError('توجد قيم مفقودة في أعمدة التنبؤ؛ افحصها قبل التدريب.')
if len(forecast_train) != len(df) or forecast_train['num_orders'].sum() != df['num_orders'].sum():
    raise AssertionError('صفوف train.csv أو مجموع num_orders لا يطابقان البيانات المدمجة المستخدمة في التحليل.')
forecast_keys = ['center_id', 'meal_id', 'week']
if forecast_train.duplicated(forecast_keys).any():
    raise ValueError('مفتاح المركز-الوجبة-الأسبوع مكرر؛ لا يمكن تقييم التوقع على هذا المستوى بأمان.')

observed_weeks = sorted(forecast_train['week'].unique())
if len(observed_weeks) < 11:
    raise ValueError('لا توجد أسابيع كافية لحجز أحدث 10 أسابيع مع فترة تدريب سابقة.')
validation_weeks = observed_weeks[-10:]
first_validation_week = validation_weeks[0]
model_training = forecast_train.loc[forecast_train['week'] < first_validation_week].copy()
validation = forecast_train.loc[forecast_train['week'].isin(validation_weeks)].copy()
if model_training.empty or validation.empty:
    raise ValueError('تقسيم التدريب والتحقق الزمني فارغ.')
print(f'صفوف التدريب: {len(model_training):,}؛ الأسابيع {model_training.week.min()}–{model_training.week.max()}')
print(f'صفوف فترة التحقق: {len(validation):,}؛ الأسابيع {validation_weeks[0]}–{validation_weeks[-1]}')
print(f'صفوف التدريب الأصلية: {len(forecast_train):,}؛ لم تُعدّل ملفات data/raw/.')

صفوف التدريب: 423,727؛ الأسابيع 1–135
صفوف فترة التحقق: 32,821؛ الأسابيع 136–145
صفوف التدريب الأصلية: 456,548؛ لم تُعدّل ملفات data/raw/.


وعد فؤاد: ترميز ميزات Linear Regression وتقييمه مقابل توقع الأسبوع السابق على الصفوف المشتركة.


In [15]:
forecast_numeric_features = [
    'week', 'checkout_price', 'base_price',
    'emailer_for_promotion', 'homepage_featured',
]
forecast_categorical_features = ['center_id', 'meal_id']

def fit_forecast_encoder(frame):
    means = frame[forecast_numeric_features].astype(float).mean()
    scales = frame[forecast_numeric_features].astype(float).std(ddof=0).replace(0, 1)
    categories = {
        column: sorted(frame[column].astype(str).unique())
        for column in forecast_categorical_features
    }
    return {'means': means, 'scales': scales, 'categories': categories}

def transform_forecast_features(frame, encoder):
    numeric = (frame[forecast_numeric_features].astype(float) - encoder['means']) / encoder['scales']
    blocks = [
        csr_matrix(np.ones((len(frame), 1), dtype=float)),
        csr_matrix(numeric.to_numpy(dtype=float)),
    ]
    for column in forecast_categorical_features:
        levels = encoder['categories'][column]
        category_to_column = {value: position for position, value in enumerate(levels[1:])}
        encoded = frame[column].astype(str).map(category_to_column)
        present = encoded.notna()
        rows = np.flatnonzero(present.to_numpy())
        columns = encoded.loc[present].astype(int).to_numpy()
        block = csr_matrix(
            (np.ones(len(rows), dtype=float), (rows, columns)),
            shape=(len(frame), max(len(levels) - 1, 0)),
        )
        blocks.append(block)
    return hstack(blocks, format='csr')

def fit_linear_demand_model(frame, target):
    encoder = fit_forecast_encoder(frame)
    matrix = transform_forecast_features(frame, encoder)
    solution = lsqr(matrix, target.to_numpy(dtype=float), atol=1e-8, btol=1e-8, iter_lim=1000)
    if solution[1] not in (1, 2, 4, 5):
        raise RuntimeError(f'لم يتقارب حل الانحدار الخطي؛ رمز LSQR={solution[1]} بعد {solution[2]} تكرارًا.')
    return {'encoder': encoder, 'coefficients': solution[0], 'iterations': solution[2], 'stop_code': solution[1]}

def predict_linear_demand(model, frame):
    return transform_forecast_features(frame, model['encoder']) @ model['coefficients']

validation_model = fit_linear_demand_model(model_training, model_training['num_orders'])
validation['توقع_الانحدار_الخطي_غير_المقيد'] = predict_linear_demand(validation_model, validation)
validation['توقع_الانحدار_الخطي'] = validation['توقع_الانحدار_الخطي_غير_المقيد'].clip(lower=0)

previous_week = forecast_train[forecast_keys + ['num_orders']].copy()
previous_week['week'] = previous_week['week'] + 1
previous_week = previous_week.rename(columns={'num_orders': 'طلب_الأسبوع_السابق'})
validation['طلب_فعلي'] = validation['num_orders']
matched_forecasts = validation.merge(
    previous_week,
    on=forecast_keys,
    how='inner',
    validate='one_to_one',
)
if matched_forecasts.empty:
    raise ValueError('لا توجد صفوف تحقق لها طلب مسجل في الأسبوع السابق؛ لا يمكن مقارنة MAE.')
matched_forecasts['الخطأ_المطلق_للنموذج'] = (
    matched_forecasts['طلب_فعلي'] - matched_forecasts['توقع_الانحدار_الخطي']
).abs()
matched_forecasts['الخطأ_المطلق_لخط_الأساس'] = (
    matched_forecasts['طلب_فعلي'] - matched_forecasts['طلب_الأسبوع_السابق']
).abs()
if not matched_forecasts['week'].isin(validation_weeks).all():
    raise AssertionError('تسربت أسابيع خارج فترة التحقق إلى حساب MAE.')

linear_mae = float(matched_forecasts['الخطأ_المطلق_للنموذج'].mean())
previous_week_mae = float(matched_forecasts['الخطأ_المطلق_لخط_الأساس'].mean())
if not np.isfinite(matched_forecasts['توقع_الانحدار_الخطي']).all():
    raise ValueError('توجد توقعات غير منتهية في فترة التحقق.')
forecast_mae_summary = pd.DataFrame({
    'الطريقة': ['Linear Regression', 'خط أساس: طلب الأسبوع السابق'],
    'MAE (وحدة طلب لكل وجبة ومركز وأسبوع)': [linear_mae, previous_week_mae],
    'عدد صفوف التقييم المشتركة': [len(matched_forecasts), len(matched_forecasts)],
})
forecast_mae_summary['تحسن النموذج مقابل خط الأساس (%)'] = [
    (previous_week_mae - linear_mae) / previous_week_mae * 100 if previous_week_mae else np.nan,
    np.nan,
]
display(forecast_mae_summary)
if linear_mae > previous_week_mae:
    print(f'في فترة التحقق هذه، MAE للنموذج أعلى من خط الأساس بنسبة {(linear_mae - previous_week_mae) / previous_week_mae * 100:.2f}٪؛ لا يُفضّل اعتماده بدل خط الأساس دون تحسين واختبار إضافي.')
else:
    print(f'في فترة التحقق هذه، MAE للنموذج أقل من خط الأساس بنسبة {(previous_week_mae - linear_mae) / previous_week_mae * 100:.2f}٪.')
print(f'الصفوف المستبعدة من المقارنة لغياب الطلب المسجل في الأسبوع السابق: {len(validation) - len(matched_forecasts):,} من {len(validation):,}.')
print(f'توقعات النموذج السالبة التي حُوّلت إلى صفر: {(validation["توقع_الانحدار_الخطي_غير_المقيد"] < 0).sum():,} من {len(validation):,} في التحقق.')
print(f'الانحدار الخطي: ميزات رقمية موحدة + ترميز one-hot للمركز والوجبة؛ LSQR تقارب بعد {validation_model["iterations"]} تكرارًا.')

forecast_detail_columns = [
    'week', 'center_id', 'meal_id', 'طلب_فعلي',
    'توقع_الانحدار_الخطي_غير_المقيد', 'توقع_الانحدار_الخطي', 'طلب_الأسبوع_السابق',
    'الخطأ_المطلق_للنموذج', 'الخطأ_المطلق_لخط_الأساس',
]
validation_forecast_table = matched_forecasts[forecast_detail_columns].sort_values(
    ['week', 'center_id', 'meal_id']
).reset_index(drop=True)
display(validation_forecast_table.head(20))
weekly_forecast_summary = (
    matched_forecasts.groupby('week', sort=True)
    .agg(
        عدد_الصفوف=('طلب_فعلي', 'size'),
        الطلب_الفعلي=('طلب_فعلي', 'sum'),
        مجموع_توقع_النموذج=('توقع_الانحدار_الخطي', 'sum'),
        MAE_النموذج=('الخطأ_المطلق_للنموذج', 'mean'),
        مجموع_توقع_الأسبوع_السابق=('طلب_الأسبوع_السابق', 'sum'),
        MAE_خط_الأسبوع_السابق=('الخطأ_المطلق_لخط_الأساس', 'mean'),
    )
)
display(weekly_forecast_summary)

,الطريقة,MAE (وحدة طلب لكل وجبة ومركز وأسبوع),عدد صفوف التقييم المشتركة,تحسن النموذج مقابل خط الأساس (%)
0,Linear Regression,126.609256,31245,-54.811568
1,خط أساس: طلب الأسبوع السابق,81.782813,31245,NaN


في فترة التحقق هذه، MAE للنموذج أعلى من خط الأساس بنسبة 54.81٪؛ لا يُفضّل اعتماده بدل خط الأساس دون تحسين واختبار إضافي.
الصفوف المستبعدة من المقارنة لغياب الطلب المسجل في الأسبوع السابق: 1,576 من 32,821.
توقعات النموذج السالبة التي حُوّلت إلى صفر: 6,405 من 32,821 في التحقق.
الانحدار الخطي: ميزات رقمية موحدة + ترميز one-hot للمركز والوجبة؛ LSQR تقارب بعد 46 تكرارًا.


,week,center_id,meal_id,طلب_فعلي,توقع_الانحدار_الخطي_غير_المقيد,توقع_الانحدار_الخطي,طلب_الأسبوع_السابق,الخطأ_المطلق_للنموذج,الخطأ_المطلق_لخط_الأساس
0,136,10,1062,918,642.120521,642.120521,960,275.879479,42
1,136,10,1109,1621,804.070790,804.070790,1554,816.929210,67
2,136,10,1198,432,372.468653,372.468653,188,59.531347,244
3,136,10,1207,216,352.940326,352.940326,257,136.940326,41
4,136,10,1216,107,252.313240,252.313240,215,145.313240,108
5,136,10,1230,123,354.894552,354.894552,122,231.894552,1
6,136,10,1247,27,252.347370,252.347370,14,225.347370,13
7,136,10,1248,405,318.147677,318.147677,203,86.852323,202
8,136,10,1311,784,481.860876,481.860876,594,302.139124,190
9,136,10,1438,54,253.564458,253.564458,134,199.564458,80


,عدد_الصفوف,الطلب_الفعلي,مجموع_توقع_النموذج,MAE_النموذج,مجموع_توقع_الأسبوع_السابق,MAE_خط_الأسبوع_السابق
week,,,,,,
136,3108,764644,878070.445984,132.751931,771939,84.350386
137,3119,712846,810031.957667,129.115735,764564,82.916961
138,3127,662792,777751.790999,123.508917,713631,75.385673
139,3113,745593,763111.695893,118.182962,662367,77.531642
140,3158,796348,807006.603051,132.054888,748447,78.928752
141,3144,772138,826246.660554,120.419607,795219,99.981870
142,3090,735124,820134.583222,121.004862,770542,64.740453
143,3105,748599,761662.154954,122.817991,735527,90.014171
144,3164,797241,876956.794848,137.353829,755402,78.044564


وعد فؤاد: تدريب النموذج على بيانات التدريب كاملة وإنتاج توقعات test.csv غير المعنونة حسب الصف والأسبوع.


In [16]:
if forecast_test_path.is_file():
    forecast_test = pd.read_csv(forecast_test_path)
    test_required = forecast_required - {'num_orders'}
    missing_test_columns = sorted(test_required - set(forecast_test.columns))
    if missing_test_columns:
        raise ValueError(f'أعمدة test.csv المطلوبة غير موجودة: {missing_test_columns}')
    if forecast_test[list(test_required)].isna().any().any():
        raise ValueError('توجد قيم مفقودة في ميزات test.csv؛ افحصها قبل التنبؤ.')
    if 'num_orders' in forecast_test.columns:
        raise ValueError('توقعنا أن test.csv غير معنونة؛ تحقق من مصدر الملف قبل استخدامه.')
    if forecast_test.duplicated(forecast_keys).any():
        raise ValueError('مفتاح المركز-الوجبة-الأسبوع مكرر في test.csv.')
    if forecast_test['week'].min() <= forecast_train['week'].max():
        raise ValueError('أسابيع test.csv ليست كلها مستقبلية مقارنة بالتدريب؛ أوقف التنبؤ لتجنب التسرب.')

    final_linear_model = fit_linear_demand_model(forecast_train, forecast_train['num_orders'])
    forecast_test['توقع_الطلبات_غير_المقيدة'] = predict_linear_demand(final_linear_model, forecast_test)
    forecast_test['توقع_الطلبات'] = forecast_test['توقع_الطلبات_غير_المقيدة'].clip(lower=0)
    test_forecast_columns = [
        column for column in ['id', 'week', 'center_id', 'meal_id']
        if column in forecast_test.columns
    ] + ['توقع_الطلبات']
    test_forecast_table = forecast_test[test_forecast_columns].sort_values(
        ['week', 'center_id', 'meal_id']
    ).reset_index(drop=True)
    print(f'توقع test.csv: {len(test_forecast_table):,} صفًا، للأسابيع {forecast_test.week.min()}–{forecast_test.week.max()}؛ حُوّلت {(forecast_test["توقع_الطلبات_غير_المقيدة"] < 0).sum():,} توقعات سالبة إلى صفر.')
    print('قُدّر نموذج test.csv على كامل الأسابيع المعنونة في train.csv؛ لا يحتوي test.csv على num_orders، لذا لا يُحسب له MAE.')
    display(test_forecast_table.head(20))
    test_forecast_weekly_summary = (
        forecast_test.groupby('week', sort=True)
        .agg(
            عدد_صفوف_التوقع=('توقع_الطلبات', 'size'),
            مجموع_الطلب_المتوقع=('توقع_الطلبات', 'sum'),
            متوسط_الطلب_المتوقع_لكل_صف=('توقع_الطلبات', 'mean'),
        )
    )
    display(test_forecast_weekly_summary)
else:
    test_forecast_table = None
    test_forecast_weekly_summary = None
    print(f'لم يُعثر على test.csv في {forecast_test_path}؛ لم تُنشأ توقعات اختبار.')

assert matched_forecasts['الخطأ_المطلق_للنموذج'].notna().all()
assert matched_forecasts['الخطأ_المطلق_لخط_الأساس'].notna().all()
assert len(validation_forecast_table) == len(matched_forecasts)
assert forecast_train['num_orders'].sum() == df['num_orders'].sum()
assert np.isfinite(forecast_test['توقع_الطلبات']).all() if test_forecast_table is not None else True
assert set(matched_forecasts['week'].unique()).issubset(set(validation_weeks))
print('نجحت فحوص MAE المشتركة، وحدود الأسابيع، ومجموع الطلبات في train.csv.')

توقع test.csv: 32,573 صفًا، للأسابيع 146–155؛ حُوّلت 6,458 توقعات سالبة إلى صفر.
قُدّر نموذج test.csv على كامل الأسابيع المعنونة في train.csv؛ لا يحتوي test.csv على num_orders، لذا لا يُحسب له MAE.


,id,week,center_id,meal_id,توقع_الطلبات
0,1177098,146,10,1062,644.992136
1,1287288,146,10,1109,744.668252
2,1386036,146,10,1198,404.044844
3,1391435,146,10,1207,347.994491
4,1330467,146,10,1216,255.707786
5,1279057,146,10,1230,363.133197
6,1399101,146,10,1248,271.213137
7,1002589,146,10,1311,477.682522
8,1225988,146,10,1438,255.174564
9,1467437,146,10,1445,196.739637


,عدد_صفوف_التوقع,مجموع_الطلب_المتوقع,متوسط_الطلب_المتوقع_لكل_صف
week,,,
146,3240,742987.616498,229.317166
147,3271,798350.547220,244.069259
148,3332,840845.136338,252.354483
149,3313,881559.751927,266.091081
150,3277,917901.319295,280.104156
151,3224,887379.133921,275.241667
152,3242,807351.563179,249.028860
153,3268,776352.297655,237.561903
154,3228,742977.137599,230.166400


نجحت فحوص MAE المشتركة، وحدود الأسابيع، ومجموع الطلبات في train.csv.


### حدود التنبؤ وكيفية استخدامه

### ما الذي تقوله نتائج التحقق؟

على الأسابيع 136–145، بلغ MAE للانحدار الخطي **126.61 طلبًا لكل صف وجبة-مركز-أسبوع**، مقابل **81.78** لخط الأساس؛ أي إن خطأ النموذج أعلى بنحو **54.81%**. حُسب المقياسان على **31,245 صفًا مشتركة** من أصل **32,821** صف تحقق؛ واستُبعدت **1,576** صفًا لغياب الطلب المسجل في الأسبوع السابق. لذلك لا تدعم هذه النتيجة اعتماد الانحدار الخطي بدل خط الأساس دون تحسين واختبار إضافي.

خط الأساس متجدد: يتنبأ بطلب كل زوج مركز-وجبة في الأسبوع باستخدام الطلب الفعلي لذلك الزوج في الأسبوع السابق. وهو مرجع قصير الأجل، وليس توقعًا مستقلًا لعشرة أسابيع مقدمًا. أما نموذج الانحدار الخطي فدُرّب على الأسابيع 1–135 مرة واحدة في هذا التحقق؛ ومن ميزاته الأسبوع والأسعار ومؤشرا الترويج ومعرّفا المركز والوجبة. يفترض استخدام الأسعار والترويج للأسبوع المستهدف أن تكون هذه المعلومات معروفة وقت التخطيط. وحُوّلت التوقعات السالبة إلى صفر: **6,405** صفوف في التحقق و**6,458** صفًا في test.csv.

### الاستخدام العملي والحدود

تغطي توقعات `test.csv` عدد **32,573** صفًا لكل مركز ووجبة في الأسابيع **146–155**. استخدمها كإشارة أولية لمراجعة الاحتياجات، لا كطلب شراء جاهز: قارِنها بالطلب الفعلي الأحدث، وراجع خصوصًا الأزواج ذات التوقعات المرتفعة أو التذبذب التاريخي الكبير، وافحص الأزواج التي لا تتوفر لها إحصاءات تاريخية قبل اتخاذ قرار. ولأن خط الأساس حقق خطأ أقل في فترة التحقق، فليظل نقطة المقارنة التشغيلية إلى أن يثبت نموذج محسّن تفوقًا على فترات تحقق زمنية إضافية.

لا يحتوي `test.csv` على `num_orders` فعلي، لذلك لا يمكن تقييم توقعاته بـMAE بعد. ولا يحدد التنبؤ وحده كمية الشراء أو مخزون الأمان: لا تتوفر بيانات المخزون المتاح أو الهدر أو مهلة المورد أو حجم العبوة ومستوى الخدمة المطلوب، كما أن أرقام الأسابيع ليست تواريخ تقويمية. حدّث التوقع بعد وصول الطلب الفعلي، وراقب MAE على الصفوف المطابقة، ثم ادمجه مع قيود المخزون والتوريد قبل اعتماد أمر شراء.